In [ ]:
%pip install chromadb sentence-transformers langchain-chroma

In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
import os

os.chdir(r"C:\Users\varun\OneDrive\Desktop\ai_Engineer_journey")
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY").strip()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=GROQ_API_KEY
)

# Embedding model — converts text to vectors
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Setup complete.")

C:\Users\varun\AppData\Local\Temp\ipykernel_4440\4077231779.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Setup complete.


In [2]:
# Create a simple ChromaDB collection
vectorstore = Chroma(
    collection_name="test_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db"  # saves to disk
)

# Add some documents
sample_docs = [
    "Python is a high level programming language",
    "Machine learning uses algorithms to learn from data",
    "Neural networks are inspired by the human brain",
    "Deep learning is a subset of machine learning",
    "Natural language processing deals with human language"
]

# Add to vectorstore
vectorstore.add_texts(sample_docs)
print(f"Added {len(sample_docs)} documents to ChromaDB")

# Query by similarity
query = "how do computers learn patterns"
results = vectorstore.similarity_search(query, k=2)

print(f"\nQuery: '{query}'")
print("Most similar documents:")
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.page_content}")

Added 5 documents to ChromaDB

Query: 'how do computers learn patterns'
Most similar documents:
  1. Machine learning uses algorithms to learn from data
  2. Neural networks are inspired by the human brain


In [3]:
# Larger document — your AI study notes
knowledge_base = """
Machine Learning Fundamentals:
Machine learning is a subset of artificial intelligence that enables systems to learn 
from data without being explicitly programmed. The three main types are supervised 
learning, unsupervised learning, and reinforcement learning. Supervised learning uses 
labeled data to train models. Common algorithms include linear regression, decision 
trees, random forests, and support vector machines.

Deep Learning and Neural Networks:
Deep learning uses neural networks with multiple layers to learn complex patterns. 
Each layer learns increasingly abstract representations. The training process uses 
backpropagation and gradient descent to minimize loss. Key architectures include 
CNNs for images, RNNs for sequences, and Transformers for language tasks. 
Dropout and batch normalization are common regularization techniques.

Natural Language Processing:
NLP enables computers to understand and generate human language. Key tasks include 
sentiment analysis, named entity recognition, machine translation, and summarization. 
Modern NLP is dominated by transformer models like BERT and GPT. Tokenization converts 
text to numbers. Embeddings represent words and sentences as dense vectors.

Large Language Models:
Large language models like GPT and LLaMA are trained on massive text datasets using 
self-supervised learning. They use the transformer architecture with attention mechanisms. 
Fine-tuning adapts pretrained models to specific tasks. Prompt engineering guides model 
behavior without changing weights. RAG combines LLMs with external knowledge retrieval.

Model Evaluation:
Evaluating ML models requires careful metric selection. Classification uses accuracy, 
precision, recall, and F1 score. Regression uses MSE, MAE, and R-squared. Cross 
validation provides reliable performance estimates. Overfitting occurs when models 
memorize training data. Regularization techniques like dropout and L2 prevent overfitting.
"""

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=75,
    length_function=len
)

docs = [Document(page_content=knowledge_base)]
chunks = splitter.split_documents(docs)

print(f"Document split into {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk.page_content[:60]}...")

Document split into 10 chunks
Chunk 1: Machine Learning Fundamentals:
Machine learning is a subset ...
Chunk 2: labeled data to train models. Common algorithms include line...
Chunk 3: Deep Learning and Neural Networks:
Deep learning uses neural...
Chunk 4: CNNs for images, RNNs for sequences, and Transformers for la...
Chunk 5: Natural Language Processing:
NLP enables computers to unders...
Chunk 6: text to numbers. Embeddings represent words and sentences as...
Chunk 7: Large Language Models:
Large language models like GPT and LL...
Chunk 8: behavior without changing weights. RAG combines LLMs with ex...
Chunk 9: Model Evaluation:
Evaluating ML models requires careful metr...
Chunk 10: memorize training data. Regularization techniques like dropo...


In [4]:
# Create a fresh vectorstore with your knowledge base
kb_vectorstore = Chroma(
    collection_name="ai_knowledge_base",
    embedding_function=embeddings,
    persist_directory="./chroma_kb"
)

# Add all chunks
kb_vectorstore.add_documents(chunks)
print(f"Stored {len(chunks)} chunks in ChromaDB")

# Test retrieval with different queries
test_queries = [
    "What is the relationship between ML and AI?",
    "How do neural networks learn?",
    "What algorithms are used in supervised learning?",
    "How do you prevent overfitting?"
]

print("\n=== Retrieval Test ===")
for query in test_queries:
    results = kb_vectorstore.similarity_search(query, k=2)
    print(f"\nQuery: '{query}'")
    print(f"Retrieved: {results[0].page_content[:100]}...")

Stored 10 chunks in ChromaDB

=== Retrieval Test ===

Query: 'What is the relationship between ML and AI?'
Retrieved: Machine Learning Fundamentals:
Machine learning is a subset of artificial intelligence that enables ...

Query: 'How do neural networks learn?'
Retrieved: Deep Learning and Neural Networks:
Deep learning uses neural networks with multiple layers to learn ...

Query: 'What algorithms are used in supervised learning?'
Retrieved: labeled data to train models. Common algorithms include linear regression, decision 
trees, random f...

Query: 'How do you prevent overfitting?'
Retrieved: memorize training data. Regularization techniques like dropout and L2 prevent overfitting....


In [5]:
def rag_answer(query, vectorstore, llm, k=3):
    # Step 1 — Retrieve relevant chunks
    relevant_docs = vectorstore.similarity_search(query, k=k)
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    # Step 2 — Build prompt with retrieved context
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a knowledgeable AI assistant. 
        Answer the question using ONLY the provided context.
        If the answer is not in the context, say 'I don't have that information.'
        Be concise and accurate.
        
        Context:
        {context}"""),
        ("human", "{question}")
    ])

    chain = prompt | llm | StrOutputParser()

    # Step 3 — Generate answer
    answer = chain.invoke({
        "context": context,
        "question": query
    })

    return answer, relevant_docs

# Test your RAG pipeline
questions = [
    "What is the difference between supervised and unsupervised learning?",
    "What is backpropagation and how does it work?",
    "How does RAG work?",
    "What metrics should I use for classification problems?",
    "What is quantum entanglement?"  # not in knowledge base
]

print("=== Full RAG Pipeline ===\n")
for question in questions:
    answer, docs = rag_answer(question, kb_vectorstore, llm)
    print(f"Q: {question}")
    print(f"A: {answer}")
    print(f"Retrieved from: {docs[0].page_content[:60]}...")
    print()

=== Full RAG Pipeline ===

Q: What is the difference between supervised and unsupervised learning?
A: Supervised learning uses labeled data to train models, whereas unsupervised learning does not.
Retrieved from: Machine Learning Fundamentals:
Machine learning is a subset ...

Q: What is backpropagation and how does it work?
A: I don't have that information. The context only mentions that "The training process uses backpropagation and gradient descent to minimize loss" but doesn't provide further details on how backpropagation works.
Retrieved from: Deep Learning and Neural Networks:
Deep learning uses neural...

Q: How does RAG work?
A: RAG combines Large Language Models (LLMs) with external knowledge retrieval, allowing for behavior change without changing weights.
Retrieved from: behavior without changing weights. RAG combines LLMs with ex...

Q: What metrics should I use for classification problems?
A: For classification problems, you should use: 
1. Accuracy
2. Precision
3. Recall

In [6]:
from langchain_chroma import Chroma

def analyze_retrieval(query, vectorstore, k=3):
    # Get documents with similarity scores
    results = vectorstore.similarity_search_with_score(query, k=k)

    print(f"Query: '{query}'")
    print(f"Retrieved {len(results)} chunks:")
    for i, (doc, score) in enumerate(results, 1):
        # ChromaDB returns L2 distance — lower is better
        print(f"  {i}. Score: {score:.4f} | {doc.page_content[:80]}...")
    print()

# Analyze retrieval quality
queries_to_analyze = [
    "What is gradient descent?",
    "How does attention mechanism work?",
    "What is overfitting?"
]

print("=== Retrieval Quality Analysis ===\n")
for query in queries_to_analyze:
    analyze_retrieval(query, kb_vectorstore)

=== Retrieval Quality Analysis ===

Query: 'What is gradient descent?'
Retrieved 3 chunks:
  1. Score: 1.1150 | Deep Learning and Neural Networks:
Deep learning uses neural networks with multi...
  2. Score: 1.3198 | Machine Learning Fundamentals:
Machine learning is a subset of artificial intell...
  3. Score: 1.3493 | memorize training data. Regularization techniques like dropout and L2 prevent ov...

Query: 'How does attention mechanism work?'
Retrieved 3 chunks:
  1. Score: 1.2876 | Large Language Models:
Large language models like GPT and LLaMA are trained on m...
  2. Score: 1.4187 | Natural Language Processing:
NLP enables computers to understand and generate hu...
  3. Score: 1.4205 | Deep Learning and Neural Networks:
Deep learning uses neural networks with multi...

Query: 'What is overfitting?'
Retrieved 3 chunks:
  1. Score: 0.9677 | memorize training data. Regularization techniques like dropout and L2 prevent ov...
  2. Score: 1.2067 | Model Evaluation:
Evaluating ML model

In [7]:
# Rebuild simple keyword search from Day 11
def keyword_search(query, chunks, top_k=2):
    results = []
    query_words = set(query.lower().split())
    for chunk in chunks:
        chunk_words = set(chunk.page_content.lower().split())
        overlap = len(query_words & chunk_words)
        results.append((overlap, chunk.page_content))
    results.sort(reverse=True)
    return [r[1] for r in results[:top_k]]

# Compare both approaches
comparison_queries = [
    "What is the relationship between ML and AI?",
    "How do models avoid memorizing training data?",
    "What makes language models understand text?"
]

print("=== Keyword vs Vector Search Comparison ===\n")
for query in comparison_queries:
    print(f"Query: '{query}'")

    # Keyword search
    keyword_results = keyword_search(query, chunks)
    print(f"Keyword result: {keyword_results[0][:80]}...")

    # Vector search
    vector_results = kb_vectorstore.similarity_search(query, k=1)
    print(f"Vector result:  {vector_results[0].page_content[:80]}...")
    print()

=== Keyword vs Vector Search Comparison ===

Query: 'What is the relationship between ML and AI?'
Keyword result: Machine Learning Fundamentals:
Machine learning is a subset of artificial intell...
Vector result:  Machine Learning Fundamentals:
Machine learning is a subset of artificial intell...

Query: 'How do models avoid memorizing training data?'
Keyword result: memorize training data. Regularization techniques like dropout and L2 prevent ov...
Vector result:  memorize training data. Regularization techniques like dropout and L2 prevent ov...

Query: 'What makes language models understand text?'
Keyword result: Natural Language Processing:
NLP enables computers to understand and generate hu...
Vector result:  Large Language Models:
Large language models like GPT and LLaMA are trained on m...

